# 02 · Radar Room (contest exercise)

**Style C: contest simulation**, a close copy of T25-RADAR (IOAI 2025 Day 1): multi-channel radar heatmaps → a
semantic label map with background −1, the ×50 object-pixel metric, and per-pixel CSV submissions
(`filename, pixel_0 … pixel_N`) for the validation and test sets zipped together. The data is synthetic, so it
generates in a second anywhere. **Traces to** `notes/ioai-task-patterns.md` P1, P2, P8.

## 1. Problem Description

A mmWave radar produces 4 heatmaps per frame. Stationary and moving reflections have different signatures across
the channels. Label every range–angle cell as background or as one of 4 target types. Beware of **ghost
reflections**: energy that looks like an object but is not labelled.

## 2. Dataset

Each sample is `x: float32 (4, 48, 96)` in [0, 1] and `y: int64 (48, 96)` with labels
**−1** background, **0** suitcase, **1** chair, **2** human, **3** wall. Samples are named `"{i}.mat.pt"`.

| Split | Samples | Labels |
|---|---|---|
| `train` | 1,500 | ✅ |
| `validation` | 400 | hidden (Leaderboard A) |
| `test` | 400 | hidden (Leaderboard B, **final**) |

## 3. Task

Predict the label map (values −1..3) for every validation and test sample.

## 4. Submission

`submission.zip` containing `submission_val.csv` and `submission_test.csv`, each with a header
`filename,pixel_0,…,pixel_4607` and one row per sample (the flattened 48×96 map, row-major).

## 5. Scoring (T25-RADAR's formula)

$$\text{Score} = \frac{|C_{0,\text{correct}}|\cdot 1 + |C_{1,\text{correct}}|\cdot 50}{|C_0|\cdot 1 + |C_1|\cdot 50}$$

Here $C_0$ is the set of background pixels and $C_1$ the set of object pixels.

- Baseline (Leaderboard B): **0.70** (the official 3-conv baseline with plain cross-entropy)
- Reference (Leaderboard B): **0.95**, which is your target

## 6. Requirements

- ≤ 10 minutes on a T4, training included. torch only, no pretrained models.

In [ ]:
import time, os, io, zipfile, random, base64, zlib
T0 = time.time()
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
device = "cuda" if torch.cuda.is_available() else "cpu"

### Data loading (DO NOT CHANGE, DO NOT DECODE)

In [ ]:
_BLOB = ("eNqtVMtu2zAQvPsrFu2FcmRVlmwncatz00t6aIA4EISAlmibjR4GSaNWv75D0Y5fadBDdaCkfcwuZ7iU1bpRhupNtW6Ja6rXvUIs6FnxgitW+6SFKHy6S0Y3Pj0mtxNv2iM8ql5SguhA8bpoqgBJfFOaZ9iZTfG6qLb1abt1gdVSySKd3vk0fcw67wwOxAdLXlWcRUHoUxiEqGPrjlAUFT0v4Nq0a8EAsSgbbuLIYT8hewBjUwvNbEYX7ttSsjaTkYv68e2rK8+V4i1L06ErMxzb9br7zHxKw6Az3NoltsuNs97Y7/H4kDJx9i4wPOSE4yxzFReNIkmyJjCzRNs7wvau54PL7h2tiqVQmqGryPOm9JfnI/3iZamnZFbIXzVK/m5qw0uag3/tU15yrSnuHeeoHb+vNSKQRAOKvc9kzn1DH/aT9FkqoZVPaqroigy+M7pKiIHSNM6o3+VvaoldVQysOXo8L0XOPVRxa3aC+WQxjwGTo57fowftjd+lp5n/FLkBQaIs5VqLPSdCUziITprIz/ceXuw9x8HNt7u4/R5Hjr4RDtmxeYJzB/PkFEEBQZ0jDO0pi4PxGUJ39MdBeIowL5s58hlrW8DnrUefgOpRv08R+GMMkwX7trNvd/YvCQ1PUHi1PmtiL9XthdyZlc6WdULb1L6doDS/lPsaGMMgwjo6SH4hdpY6uITy3kGr5arRhpRYlJBMNjVUc/INSvkiSNRCLVucdG5Iarr//kAln0NXURAzK0EfjBJCf6BmQfa3u6tImwbs/Ousxd7RXP4fxd/SK4LWbwoWvzouFLsU4pT5yM3ZTpFCKpmvSmHY/i6EHvBFZ6IoYTaqtjdhjvlgM/QwCyq+ZXwrdWKvhhi7fBFiXchKJw9qI/APpoZ4PfX+AKEhl0U=")
exec(zlib.decompress(base64.b64decode(_BLOB)).decode())
X_train, Y_train = _radar(1500, 11)
X_val, X_test = _radar(400, 22)[0], _radar(400, 33)[0]
val_names = [f"{i}.mat.pt" for i in range(400)]; test_names = [f"{i}.mat.pt" for i in range(400)]
print(X_train.shape, Y_train.shape, X_val.shape, "background share:", (Y_train == -1).mean().round(3))

### Baseline (replace this): the official T25-RADAR model

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 16, 3, padding=1); self.conv2 = nn.Conv2d(16, 32, 3, padding=1); self.conv3 = nn.Conv2d(32, 5, 3, padding=1)
    def forward(self, x):
        return self.conv3(F.relu(self.conv2(F.relu(self.conv1(x)))))

def train_model():
    model = MyModel().to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train) + 1), batch_size=4, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(5):
        model.train()
        for xb, yb in loader:
            loss = F.cross_entropy(model(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch} loss {loss.item():.4f}")
    return model

### Train and predict

In [ ]:
model = train_model()

@torch.no_grad()
def predict(model, X, bs=100):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(device)
        logits = model(xb).softmax(1) + model(xb.flip(-1)).softmax(1).flip(-1)
        out.append(logits.argmax(1).cpu() - 1)
    return torch.cat(out).numpy()

### Local validation (official metric on held-out training maps)

In [ ]:
def radar_score(pred, truth, bonus=50):
    ok, bg = pred == truth, truth == -1
    return ((ok & bg).sum() + bonus * (ok & ~bg).sum()) / (bg.sum() + bonus * (~bg).sum())
print("score on the last 200 training maps (seen during training):", round(float(radar_score(predict(model, X_train[-200:]), Y_train[-200:])), 4))

### Submission

In [ ]:
def to_csv_bytes(names, pred):
    df = pd.DataFrame(pred.reshape(len(pred), -1), columns=[f"pixel_{i}" for i in range(pred[0].size)])
    df.insert(0, "filename", names)
    return df.to_csv(index=False).encode()
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("submission_val.csv", to_csv_bytes(val_names, predict(model, X_val)))
    z.writestr("submission_test.csv", to_csv_bytes(test_names, predict(model, X_test)))
print("wrote submission.zip")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    truth = {"submission_val.csv": _radar(400, 22)[1], "submission_test.csv": _radar(400, 33)[1]}
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, Y in truth.items():
            try:
                df = pd.read_csv(z.open(name))
                P = df[[f"pixel_{i}" for i in range(48 * 96)]].to_numpy().reshape(-1, 48, 96)
                assert len(df) == len(Y) and (df["filename"].to_numpy() == np.array([f"{i}.mat.pt" for i in range(len(Y))])).all()
                out.append(float(radar_score(P, Y)))
            except Exception as e:
                print(name, "rejected:", e); out.append(0.0)
    return out

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
elapsed = time.time() - T0
print(f"total runtime {elapsed / 60:.1f} min (limit 10 min)")
assert elapsed < 600